# Autoencoder reconstruction comparison

Compare the released MLP autoencoder with the mixed-baseline CNN on the same complete PubMed (3,066), Nilearn (79), and NeuroVault (202) test splits. Set `INCLUDE_FINETUNED = True` only when you explicitly want the domain-fine-tuned CNN branches.

The provider downloads the published root-level split JSONLs and shared volume tensor from Hugging Face. Legacy local paths stored in individual JSONL rows are metadata only and are never loaded.

This is the `paired_atlas_free` protocol. Older PubMed MLP autoencoder comparisons loaded the MLP-native PubMed image resource and selected a different first-N cohort, while the CNN used atlas-free rows. Those older values are valid for their native cohort but are not expected to match this notebook. Here both families receive the same examples; the MLP bridge converts each shared volume to its established masker flat-map representation.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import torch

from neurovlm import AtlasFreeCNNDataProvider, load_pipeline
from neurovlm.evaluation import (
    default_comparison_matrix,
    evaluate_reconstruction_comparison,
)

DOMAINS = ("pubmed", "nilearn", "neurovault")
LIMIT_PER_DOMAIN = None  # full test split; set an integer only for a quick run
INCLUDE_FINETUNED = False  # explicit opt-in
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EVALUATION_SCOPE = (
    "full test split"
    if LIMIT_PER_DOMAIN is None
    else f"first {LIMIT_PER_DOMAIN} test maps per domain"
)


## Run the comparison

The shared volume payload is about 4.2 GB and is downloaded once into the normal Hugging Face cache. Later providers reuse the cached payload.

In [ ]:
results = []
for domain in DOMAINS:
    selections = default_comparison_matrix(
        "autoencoder",
        domains=(domain,),
        include_finetuned=INCLUDE_FINETUNED,
    )
    provider = AtlasFreeCNNDataProvider(
        domain=domain,
        limit=LIMIT_PER_DOMAIN,
    )
    results.append(evaluate_reconstruction_comparison(
        selections=selections,
        provider=provider,
        device=DEVICE,
    ))

summary = pd.DataFrame(row for result in results for row in result.summary)
by_source = pd.DataFrame(row for result in results for row in result.by_source)
by_sample = pd.DataFrame(row for result in results for row in result.by_sample)
manifest = pd.DataFrame(row for result in results for row in result.manifest)

summary.sort_values(["evaluation_domain", "family", "variant"])


## Aggregate metric plots

CNN metrics are computed in native atlas-free volume space; MLP metrics are computed in the established masker flat-map space. The plots compare complete pipelines in their declared spaces, not voxel-identical representations. Lower reconstruction MSE is better; higher spatial correlation and top-5% Dice are better.

In [ ]:
resolved = summary[(summary["status"] == "resolved") & (summary["n"] > 0)].copy()
if resolved.empty:
    raise RuntimeError("No models resolved. Inspect `manifest` for checkpoint errors.")
resolved["model"] = resolved.apply(
    lambda row: f'{row["family"].upper()} · {row["variant"]}', axis=1
)

metrics = (
    ("reconstruction_mse", "Reconstruction MSE ↓"),
    ("spatial_corr", "Spatial correlation ↑"),
    ("top5_dice", "Top-5% Dice ↑"),
)
fig, axes = plt.subplots(1, len(metrics), figsize=(17, 4.5))
for ax, (metric, title) in zip(axes, metrics):
    table = resolved.pivot(index="evaluation_domain", columns="model", values=metric)
    table.plot.bar(ax=ax, rot=0)
    ax.set_title(title)
    ax.set_xlabel("Evaluation domain")
    ax.grid(axis="y", alpha=0.25)
    ax.legend(title="Model", fontsize=8)
fig.suptitle(f"Autoencoder comparison ({EVALUATION_SCOPE})")
fig.tight_layout()
plt.show()


## Per-sample distributions

These box plots expose variance hidden by the aggregate means.

In [ ]:
samples = by_sample.copy()
samples["model"] = samples.apply(
    lambda row: f'{row["family"].upper()} · {row["variant"]}', axis=1
)
samples["group"] = samples["evaluation_domain"] + "\n" + samples["model"]
group_names = list(dict.fromkeys(samples["group"]))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, metric, title in (
    (axes[0], "spatial_corr", "Per-sample spatial correlation ↑"),
    (axes[1], "top5_dice", "Per-sample top-5% Dice ↑"),
):
    values = [samples.loc[samples["group"] == name, metric].dropna() for name in group_names]
    ax.boxplot(values, labels=group_names, showmeans=True)
    ax.set_title(title)
    ax.tick_params(axis="x", labelrotation=35)
    ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
plt.show()


## Qualitative CNN reconstruction

Display the highest-activation axial slice from one native-volume input, its mixed-baseline CNN reconstruction, and the absolute error.

In [ ]:
VISUAL_DOMAIN = "pubmed"
example = AtlasFreeCNNDataProvider(domain=VISUAL_DOMAIN, limit=1).test[0]
cnn = load_pipeline(family="cnn", task="autoencoder", device=DEVICE)
truth = example["volume"][0].cpu()
reconstruction = cnn.reconstruct(example["volume"].unsqueeze(0))[0, 0].cpu()
slice_index = int(truth.abs().sum(dim=(0, 1)).argmax())
vmax = float(torch.stack((truth.max(), reconstruction.max())).max())

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
panels = (
    (truth, "Input", "hot", 0.0, vmax),
    (reconstruction, "CNN reconstruction", "hot", 0.0, vmax),
    ((reconstruction - truth).abs(), "Absolute error", "magma", 0.0, None),
)
for ax, (volume, title, cmap, vmin, panel_vmax) in zip(axes, panels):
    image = ax.imshow(
        volume[:, :, slice_index].T,
        origin="lower",
        cmap=cmap,
        vmin=vmin,
        vmax=panel_vmax,
    )
    ax.set_title(title)
    ax.axis("off")
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
fig.suptitle(f'{VISUAL_DOMAIN.title()} · {example["map_id"]} · axial slice {slice_index}')
fig.tight_layout()
plt.show()
